In [1]:
# =============================================================================
# GUS02: Cross Tables & Data Quality-of-Life Functions
# =============================================================================
# This notebook demonstrates the v4.1 cross table and QoL capabilities:
#   1. Load database with data (from GUS01F)
#   2. Fix & re-process census data (verify the parse_values fix)
#   3. Build cross tables from loaded subject data
#   4. Inspect cross tables (DataFrame view, per year)
#   5. Aggregate cross tables across TERYTs
#   6. Manually insert a cross table and deconstruct to raw data
#   7. QoL functions: subject_availability, get_subject_dataframe, etc.
#   8. Save/reload with cross table persistence
# =============================================================================

# STEP 1: Imports and Path Setup
import os
import sys
from pathlib import Path
import importlib
import gc

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

# Find the repository root
def find_repo_root(start=Path.cwd()):
    for p in [start] + list(start.parents):
        if (p / 'Code').exists() or (p / '.git').exists():
            return p
    return start

repo_root = find_repo_root()
tools_path = repo_root / 'Code' / 'tools'
if str(tools_path) not in sys.path:
    sys.path.insert(0, str(tools_path))

import geoTERYT_db as gtdb
importlib.reload(gtdb)

# Paths
data_root = repo_root.parent.parent / 'Data'
geo_root = data_root / 'Geospatial'
gus_root = data_root / 'GUS'

print(f"Repository root: {repo_root}")
print(f"GUS root: {gus_root}")
print(f"Module version attributes: CrossTable={hasattr(gtdb, 'CrossTable')}, YEAR_RANGE_FULL={hasattr(gtdb, 'YEAR_RANGE_FULL')}")

Repository root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/local_repo/LRDWI-Paper
GUS root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/GUS
Module version attributes: CrossTable=True, YEAR_RANGE_FULL=True


In [2]:
# =============================================================================
# STEP 2: Load Database (with data from GUS01F)
# =============================================================================
complete_db_path = geo_root / 'geoteryt_complete_geom_OW.pkl'
db = gtdb.load_complete_database(complete_db_path)
db.print_summary()
print(f"\nData summary: {db.get_data_summary()}")

Loading complete database from /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_complete_geom_OW.pkl...
  Database version: 3.1
  ✓ Restored geometry data for years: [2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2015, 2016, 2017, 2018, 2021, 2022, 2023]
  ✓ Loaded 4560 records
  ✓ Year range: 1999 - 2024
  ✓ Records with geometry: 3612
  ✓ Records with old_woj: 2658
GeoTERYT Database Summary (v3.0)
Total records:           4,560
Year range:              1999 - 2024
------------------------------------------------------------
Administrative levels:
  Voivodeships (2):      16
  Powiats (5):           382
  Gminas (6):            4162
------------------------------------------------------------
Change tracking:
  Records with changes:      772
  Records with level changes: 0
  Records with kind changes:  0
------------------------------------------------------------
Geometry:
  Records with geometr

In [3]:
# =============================================================================
# STEP 3: Load Source Data (BDL + Census)
# =============================================================================
df_demographic = pd.read_csv(gus_root / "data" / 'bdl_demographic_data.csv', encoding='utf-8')
df_c_1988 = pd.read_csv(gus_root / "data" / "census_data" / 'NSP1988_data.csv', encoding='utf-8')
df_c_2002 = pd.read_csv(gus_root / "data" / "census_data" / 'NSP2002_data.csv', encoding='utf-8')
df_c_2011 = pd.read_csv(gus_root / "data" / "census_data" / 'NSP2011_data.csv', encoding='utf-8')
df_c_2021 = pd.read_csv(gus_root / "data" / "census_data" / 'NSP2021_data.csv', encoding='utf-8')

df_variables = pd.read_csv(gus_root / "metadata" / 'bdl_variables_level6.csv', encoding='utf-8')
df_c_variables = pd.read_csv(gus_root / "metadata" / 'census_meta.csv', encoding='utf-8')

# Fix 1988 year typo and unify meta structure
for id, row in df_c_1988.iterrows():
    df_c_1988.at[id, 'values'] = df_c_1988.at[id, 'values'].replace(", 'year': '1998'", ", 'year': '1988'")
df_c_variables['years'] = df_c_variables['years'].str.replace("[1998]", "[1988]")
df_c_variables['n4'] = None
df_c_variables['n5'] = None

# Subject registry
subject_ids = {"BDL": [], "Census": {"1988": [], "2002": [], "2011": [], "2021": []}}
subject_ids["BDL"] = list(df_demographic['subjectId'].unique())
subject_ids["Census"]["1988"] = list(df_c_1988['subjectId'].unique())
subject_ids["Census"]["2002"] = list(df_c_2002['subjectId'].unique())
subject_ids["Census"]["2011"] = list(df_c_2011['subjectId'].unique())
subject_ids["Census"]["2021"] = list(df_c_2021['subjectId'].unique())

subject_names_dict = {
    'P1336': 'pop__sex_URsplit', 'P2137': 'pop__age_sex', 'P2914': 'pop__sex_cities',
    'P2884': 'pop__age', 'P2885': 'pop__educ', 'P2883': 'pop__sex', 'P2887': 'hh_size',
    'P2114': 'pop__age_sex', 'P2403': 'pop__age_educ', 'P2402': 'pop__sex_educ', 'P2871': 'hh_size',
    'P3304': 'pop__age_sex', 'P3311': 'pop__age_educ', 'P3309': 'pop__sex_educ',
    'P3310': 'pop__educ_URsplit', 'P3420': 'hh_size',
    'P4253': 'pop__age_sex', 'P4320': 'pop__age_educ',
    'P4345': 'pop__sex_educ_URsplit', 'P4287': 'hh_size'
}

print(f"BDL subjects: {subject_ids['BDL']}")
for yr, sids in subject_ids['Census'].items():
    print(f"Census {yr} subjects: {sids}")

BDL subjects: ['P1336', 'P2137', 'P2914']
Census 1988 subjects: ['P2884', 'P2885', 'P2883', 'P2887']
Census 2002 subjects: ['P2114', 'P2403', 'P2402', 'P2871']
Census 2011 subjects: ['P3304', 'P3311', 'P3309', 'P3310', 'P3420']
Census 2021 subjects: ['P4253', 'P4320', 'P4345', 'P4287']


In [4]:
# =============================================================================
# STEP 4: Process & Load ALL Subject Data (BDL + Census)
# =============================================================================
# This verifies the fix to process_subject_data for census data.

census_dfs = {"1988": df_c_1988, "2002": df_c_2002, "2011": df_c_2011, "2021": df_c_2021}

# Process all subjects
for s in subject_ids["BDL"]:
    print(f"Processing + Loading BDL subject: {s} ({subject_names_dict.get(s, '')})...")
    df = gtdb.GeoTERYTDatabase.process_subject_data(df_demographic, df_variables, s)
    if not df.empty:
        stats = db.load_subject_data(df, source_type='BDL', subject_id=s,
                                      subject_name=subject_names_dict.get(s, ''))

for yr, sids in subject_ids["Census"].items():
    for s in sids:
        print(f"Processing + Loading Census {yr} subject: {s} ({subject_names_dict.get(s, '')})...")
        df = gtdb.GeoTERYTDatabase.process_subject_data(census_dfs[yr], df_c_variables, s)
        if not df.empty:
            stats = db.load_subject_data(df, source_type='Census', subject_id=s,
                                          subject_name=subject_names_dict.get(s, ''))

# Summary after loading
summary = db.get_data_summary()
print(f"\n{'='*60}")
print(f"Total records with data: {summary['records_with_data']}")
print(f"Total subjects loaded: {summary['n_subjects']}")
print(f"Total data series: {summary['total_data_series']:,}")
print(f"Total data points: {summary['total_data_points']:,}")

Processing + Loading BDL subject: P1336 (pop__sex_URsplit)...
  ✓ Loaded 2,298,528 data points for subject P1336
  ✓ Matched 4569 TERYT records, 10 unmatched
  ⚠ Unmatched TERYT IDs (first 10): ['0000000', '0216001', '0410001', '1207132', '1210001', '1431981', '1431991', '1465158', '1465998', '2002162']
Processing + Loading BDL subject: P2137 (pop__age_sex)...
  ✓ Loaded 7,338,336 data points for subject P2137
  ✓ Matched 4548 TERYT records, 8 unmatched
  ⚠ Unmatched TERYT IDs (first 10): ['0000000', '0216001', '0410001', '1210001', '1431981', '1431991', '1465158', '1465998']
Processing + Loading BDL subject: P2914 (pop__sex_cities)...
  ✓ Loaded 1,505,856 data points for subject P2914
  ✓ Matched 4548 TERYT records, 8 unmatched
  ⚠ Unmatched TERYT IDs (first 10): ['0000000', '0216001', '0410001', '1210001', '1431981', '1431991', '1465158', '1465998']
Processing + Loading Census 1988 subject: P2884 (pop__age)...
  ✓ Loaded 28,992 data points for subject P2884
  ✓ Matched 3624 TERYT rec

In [ ]:
# =============================================================================
# STEP 5: Build Cross Tables from Loaded Data
# =============================================================================
# Build cross tables for all subjects that have multiple variables.
# Cross tables are M-dimensional arrays (e.g., age × sex for P2137).

# Reload module to pick up any fixes
importlib.reload(gtdb)

for s, name in subject_names_dict.items():
    n_built = db.build_cross_tables(s, subject_name=name)
    
print(f"\n{'='*60}")
print("Cross table summary:")
ct_summary = db.get_cross_table_summary()
display(ct_summary)

KeyError: 2025

In [ ]:
# =============================================================================
# STEP 6: Inspect Cross Tables on Individual Records
# =============================================================================
# Example: Kraków's P2137 (pop by age × sex) cross table

krakow = [r for r in db._records.values() if 'Kraków' in r.name and r.level == 6]
if krakow:
    rec = krakow[0]
    print(f"Record: {rec}")
    print(f"Cross tables: {rec.list_cross_tables()}")
    
    ct = rec.get_cross_table('P2137')
    if ct:
        print(f"\n{ct}")
        print(f"Dimensions: {ct.dim_names}")
        print(f"Labels: {ct.dim_labels}")
        print(f"Years with data: {ct.years_with_data}")
        
        # Show as DataFrame for 2020
        print(f"\n--- Cross table for Kraków, P2137, year 2020 ---")
        display(ct.get_as_dataframe(2020))

In [ ]:
# =============================================================================
# STEP 7: Inspect Census Cross Tables (single-year data)
# =============================================================================
# Census data only has one year per subject. Let's check P2883 (pop by sex, 1988)
# and P2114 (pop by age × sex, 2002)

if krakow:
    rec = krakow[0]
    
    # 1988 census: P2883 (pop by sex)
    ct_1988 = rec.get_cross_table('P2883')
    if ct_1988:
        print(f"P2883 (1988 census - pop by sex): {ct_1988}")
        print(f"Years with data: {ct_1988.years_with_data}")
        display(ct_1988.get_as_dataframe(1988))
    else:
        print("P2883 cross table not built (might have < 2 variables)")
    
    # 2002 census: P2114 (pop by age × sex)
    ct_2002 = rec.get_cross_table('P2114')
    if ct_2002:
        print(f"\nP2114 (2002 census - pop by age × sex): {ct_2002}")
        print(f"Years with data: {ct_2002.years_with_data}")
        display(ct_2002.get_as_dataframe(2002))
    else:
        print("P2114 cross table not built")

In [ ]:
# =============================================================================
# STEP 8: Aggregate Cross Tables Across TERYTs
# =============================================================================
# Aggregate P2137 (pop by age × sex) across all gminas in Małopolskie

malopolskie_gminas = db.get_gminas_in_voivodeship('12', year=2020)
teryt_ids = [r.teryt_id for r in malopolskie_gminas]
print(f"Aggregating {len(teryt_ids)} gminas in Małopolskie voivodeship...")

agg_ct = db.aggregate_cross_tables(teryt_ids, 'P2137')
if agg_ct:
    print(f"\nAggregated: {agg_ct}")
    print(f"\n--- Aggregated cross table for Małopolskie, P2137, year 2020 ---")
    display(agg_ct.get_as_dataframe(2020))
else:
    print("No cross tables found for aggregation")

In [ ]:
# =============================================================================
# STEP 9: Manual Cross Table Insertion and Deconstruction
# =============================================================================
# Demonstrate: manually insert a cross table, then deconstruct it back to data

# Create a small hypothetical cross table for a fake subject on Kraków
if krakow:
    rec = krakow[0]
    
    # Insert a 2D cross table manually (e.g., education × sex)
    manual_table = np.array([
        [100, 200, 300],  # primary: total, male, female
        [150, 250, 350],  # secondary
        [200, 300, 400],  # tertiary
    ], dtype=float)
    
    rec.insert_cross_table_year(
        subject_id='XTEST',
        year=2020,
        table=manual_table,
        subject_name='test_educ_sex',
        dim_names=['n1', 'n2'],
        dim_labels={'n1': ['primary', 'secondary', 'tertiary'],
                    'n2': ['total', 'male', 'female']}
    )
    
    ct_manual = rec.get_cross_table('XTEST')
    print(f"Manually inserted: {ct_manual}")
    display(ct_manual.get_as_dataframe(2020))
    
    # Deconstruct back to raw data points
    n_pts = rec.deconstruct_cross_table('XTEST', year=2020, source_type='Manual')
    print(f"\nDeconstructed to {n_pts} data points")
    
    # Show the data points that were created
    manual_series = rec.get_data_by_subject('XTEST')
    for key, series in list(manual_series.items())[:5]:
        print(f"  {key}: categories={series.categories}, val@2020={series.get_value(2020)}")
    print(f"  ... ({len(manual_series)} series total)")

In [ ]:
# =============================================================================
# STEP 10: QoL - Subject Availability
# =============================================================================
# Check which subjects are available across all gminas

avail = db.subject_availability(level=6, mode='bool')
print(f"Availability matrix shape: {avail.shape}")
print(f"Columns: {list(avail.columns)}\n")

# Show summary: how many gminas have data for each subject
print("Records with data per subject:")
for col in avail.columns:
    if col == 'name':
        continue
    n_true = avail[col].sum() if avail[col].dtype == bool else (avail[col] != '').sum()
    print(f"  {col} ({subject_names_dict.get(col, '')}): {n_true}")

# Show first few rows
display(avail.head(10))

In [ ]:
# =============================================================================
# STEP 11: QoL - Get Subject DataFrame
# =============================================================================
# Reconstruct a full flat DataFrame from loaded subject data (reverse of load_subject_data)

# Example: get all P2137 data for a single TERYT
df_single = db.get_subject_dataframe('P2137', teryt_id='1201011')
print(f"P2137 data for Kraków (1201011), all years:")
print(f"  Shape: {df_single.shape}")
display(df_single.head(20))

# Get for a specific year across all records
df_year = db.get_subject_dataframe('P2137', year=2020)
print(f"\nP2137 data for year 2020, all records:")
print(f"  Shape: {df_year.shape}")
display(df_year.head(10))

In [ ]:
# =============================================================================
# STEP 12: QoL - Get Variable Values (quick population table)
# =============================================================================
# Get all values of P2137 for year 2020 across gminas

pop_2020 = db.get_variable_values('P2137', year=2020, level=6)
print(f"Population data for year 2020, gmina level:")
print(f"  Shape: {pop_2020.shape}")
print(f"  Columns: {list(pop_2020.columns)[:10]}...")
display(pop_2020.head(10))

In [ ]:
# =============================================================================
# STEP 13: Save & Reload with Cross Table Persistence
# =============================================================================
# Clean up the test cross table before saving
if krakow:
    # Remove the test cross table
    rec = krakow[0]
    if 'XTEST' in rec.cross_tables:
        del rec.cross_tables['XTEST']
    # Also remove test data series
    test_keys = [k for k in rec.data.keys() if k[1] == 'XTEST']
    for k in test_keys:
        del rec.data[k]

save_path = geo_root / 'geoteryt_complete_final.pkl'
db.save_complete(save_path)

# Reload and verify
db2 = gtdb.load_complete_database(save_path)
summary2 = db2.get_data_summary()
print(f"\nAfter reload: {summary2['records_with_data']} records with data, "
      f"{summary2['total_data_points']:,} total points")

# Check cross tables survived
ct_summary2 = db2.get_cross_table_summary()
print(f"\nCross tables after reload:")
display(ct_summary2)

# Spot check: Kraków P2137
krakow2 = [r for r in db2._records.values() if 'Kraków' in r.name and r.level == 6]
if krakow2:
    ct_check = krakow2[0].get_cross_table('P2137')
    if ct_check:
        print(f"\nKraków P2137 after reload: {ct_check}")
        display(ct_check.get_as_dataframe(2020))

gc.collect()